### Import dependencies

In [ ]:
# Model parameters used throughout the notebooks
k_on_0 = 3.47e-4
k_off_0 = 2.68e-4 
f_1e = 2.49e-1

# import utility functions
include("src/utils.jl");

# import functions that help calculate stability equilibria
include("src/f_vectors_k_vectors_nvectors.jl");
# This function return variables that act as look-up tables for stability that can be used by functions of f_vectors_k_vectors_nvectors.jl
K, F, n_stable_store, n_unstable_store = setup_stores(k_off_0 / k_on_0);

# import functions that define loading patterns
include("src/loading_patterns.jl");

# import functions that define the analytical model
include("src/analytical_model.jl");

# import functions that define the stochastic model
include("src/stochastic_model.jl");

Number of threads: 20


### High and low tensions of maps

In [ ]:
gamma_high = collect(range(0.09, 0.12, length=2)) 
gamma_low = collect(range(0.00, 0.0525, length=4)); 

## analytical maps ~ runtime approx 10 min

In [16]:
duration = 20000

# get time to recovery
time_to_recovery = get_recovery_time.(gamma_low, k_on_0, k_off_0, f_1e, duration, true)

# get time to fracture
time_to_fracture = get_fracture_time.(gamma_high, k_on_0, k_off_0, f_1e, duration)

# proportional ranges of high and low time periods relative to their respective timescales
time_high_ratio = range(0.01, 1.5, length=40)
time_low_ratio = range(0.01, 4.5, length=40)

# run simulations over grid of time ratios
for g_h in 1:length(gamma_high)
    for g_l in 1:length(gamma_low)
        # run cyclic simulations over grid of time ratios
        cyclic_time_to_fracture = zeros(length(time_high_ratio), length(time_low_ratio))
        for t1 in 1:length(time_high_ratio)
            @threads for t2 in 1:length(time_low_ratio)
                # caluclate duration to spend 20x time_to_fracture at high tension for current t_high and t_low
                t_high = time_high_ratio[t1]*time_to_fracture[g_h]
                t_low = time_low_ratio[t2]*time_to_recovery[g_l]
                objective_time_high = 20 * time_to_fracture[g_h]
                period = t_high + t_low
                n_periods = ceil(objective_time_high / t_high)
                duration = n_periods * period

                loading = square_cycle_uneven_new(gamma_high[g_h], gamma_low[g_l], t_high, t_low)
                n_init = k_on_0/(k_on_0+k_off_0)
                sol = solve_dn_dt(n_init, k_on_0, k_off_0, f_1e, loading, duration, 1.0)
                time_of_fracture = find_fracture(sol) 
                cyclic_time_to_fracture[t1, t2] = fracture_to_t_high(time_of_fracture, t_high, t_low)                          
            end
        end

        # rescale data from 0 to 1
        cyclic_time_to_fracture_plot = cyclic_time_to_fracture'
        cyclic_time_to_fracture_plot = (cyclic_time_to_fracture_plot .- time_to_fracture[g_h]) ./ cyclic_time_to_fracture_plot

        # Plot heatmap
        x = vec(time_high_ratio)
        y = vec(time_low_ratio)
        plot(size=(450, 400), guidefontsize=16, tickfontsize=14, legendfontsize=14, margin=3Plots.mm)
        heatmap!(x,y, cyclic_time_to_fracture_plot, color=:viridis, xlabel="\$t_{high}/t^*\$", ylabel="\$t_{low}/t_r\$", cbar=true, clims=(0, 1))
        xlims!(0, maximum(x))
        ylims!(0, maximum(y))

        # save figure
        savefig("figures/Fig4_analytical_gamma_high_$(round(gamma_high[g_h], digits=3))_gamma_low_$(round(gamma_low[g_l], digits=3)).svg")
    end
end

## stochastic maps ~ runtime several days

In [ ]:
duration = 20000
num_sim = 100
n = 100
l = 1

time_high_ratio = range(0.01, 1.5, length=40)
time_low_ratio = range(0.01, 4.5, length=40)

time_to_recovery_stochastic = get_recovery_time_stochastic.(gamma_low, k_on_0, k_off_0, f_1e, duration, true, false, 1.0, num_sim, n)
time_to_fracture_stochastic = get_fracture_time_stochastic.(gamma_high, k_on_0, k_off_0, f_1e, duration, nothing, num_sim, n, true)
    
for g_h in 1:length(gamma_high)
    for g_l in 1:length(gamma_low)
        cyclic_time_to_fracture_stochastic = zeros(length(time_high_ratio), length(time_low_ratio), num_sim)
        for t1 in 1:length(time_high_ratio)
            @threads for t2 in 1:length(time_low_ratio)
                # get duration to spend 20x time_to_fracture at high tension for current t_high and t_low
                objective_time_high = 20 * time_to_fracture_stochastic[g_h]
                t_high = time_high_ratio[t1]*time_to_fracture_stochastic[g_h]
                t_low = time_low_ratio[t2]*time_to_recovery_stochastic[g_l]
                period = t_high + t_low
                n_periods = ceil(objective_time_high / t_high)
                duration = n_periods * period

                # set dt to 1/100 of the minimum of time_high and time_low periods for fast simulations
                dt = minimum([t_high, t_low])/100

                # modify dt so that a multiple of dt is equal to the duration
                dt = duration/round(Int, duration/dt)
                
                # define cyclic loading for stochastic model
                loading = square_cycle_uneven_new(gamma_high[g_h], gamma_low[g_l], t_high, t_low)
                tension_bond, n_timesteps = loading_stochastic_model(loading, n, duration, dt)

                n_init = k_on_0/(k_on_0+k_off_0)
                
                for sim = 1:1:num_sim   # number of simulations
                    model = SlipBondModel((k_on_0=k_on_0,), (k_off_0=k_off_0, f_1e=f_1e)) 
                    x = Cluster(n, l, model, :force_global)
                    x = initialise_bonds_state(x, n_init)
                    _, _, fracture_time, _ = runcluster(x, tension_bond, dt, max_steps = n_timesteps, verbose=false)
                    cyclic_time_to_fracture_stochastic[t1, t2, sim] = fracture_to_t_high(fracture_time, t_high, t_low)
                end 
            end
        end
        # average over simulations
        cyclic_time_to_fracture_stochastic_avg = mean(cyclic_time_to_fracture_stochastic, dims=3)[:,:,1]

        # rescale data from 0 to 1
        cyclic_time_to_fracture_plot = cyclic_time_to_fracture_stochastic_avg'
        cyclic_time_to_fracture_plot = (cyclic_time_to_fracture_plot .- time_to_fracture_stochastic[g_h]) ./ cyclic_time_to_fracture_plot

        # Plot heatmap
        x = vec(time_high_ratio)
        y = vec(time_low_ratio)
        plot(size=(450, 400), guidefontsize=16, tickfontsize=14, legendfontsize=14, margin=3Plots.mm)
        heatmap!(x,y, cyclic_time_to_fracture_plot, color=:viridis, xlabel="\$t_{high}/t^*\$", ylabel="\$t_{low}/t_r\$", cbar=true, clims=(0, 1))
        xlims!(0, maximum(x))
        ylims!(0, maximum(y))

        # save figure
        savefig("figures/Fig4_stochastic_gamma_high_$(round(gamma_high[g_h], digits=3))_gamma_low_$(round(gamma_low[g_l], digits=3)).svg")
        println("Finished running $(100*(g_l + g_h*length(gamma_low)) /(length(gamma_high)*length(gamma_low)))")
    end
end